# 08 — Аугментация данных для редких классов

**Проблема:** disgust (0.8%), fear (1.5%), anger (5.2%) катастрофически недопредставлены.

**Два метода аугментации:**
- **Парафраз** (`cointegrated/rut5-base-paraphraser`) — diverse beam search, 3 варианта на предложение
- **Обратный перевод** (`Helsinki-NLP RU→EN→RU`) — другой словарь через английский pivot

**Результат:** `stage1_data_augmented/` — сбалансированный датасет для ноутбука 07.

In [ ]:
import sys, os

PROJECT_ROOT = '/kaggle/input/sentiment-analysis' if os.path.exists('/kaggle') else os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle') else '../results'

import torch
import pandas as pd
import matplotlib.pyplot as plt

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
!pip install -q sentencepiece sacremoses

## 1. Загрузка Stage-1 датасета

In [ ]:
from datasets import load_from_disk
from src.data_loader import EKMAN_LABEL_NAMES, EKMAN_ID2LABEL

S1_DATA_PATH = f'{WORKING_DIR}/stage1_data'
AUG_DATA_PATH = f'{WORKING_DIR}/stage1_data_augmented'

if not os.path.exists(S1_DATA_PATH):
    raise RuntimeError(
        f'Stage-1 датасет не найден: {S1_DATA_PATH}\n'
        'Сначала запусти ноутбук 07 (ячейки загрузки данных до ячейки обучения).'
    )

ds = load_from_disk(S1_DATA_PATH)
print('Stage-1 загружен.')
for split in ds:
    print(f'  {split:12s}: {len(ds[split]):,}')

In [ ]:
# Текущее распределение в train
train_df = ds['train'].to_pandas()
counts = train_df['label'].value_counts().sort_index()

EMOTION_COLORS = {
    'anger':'#e74c3c','disgust':'#8e44ad','fear':'#2c3e50',
    'joy':'#f39c12','sadness':'#3498db','surprise':'#1abc9c','neutral':'#95a5a6',
}

print('Распределение классов в train (ДО аугментации):')
for lid, cnt in counts.items():
    bar = '█' * int(cnt / counts.max() * 40)
    pct = cnt / len(train_df) * 100
    print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>6,}  ({pct:4.1f}%)  {bar}')

fig, ax = plt.subplots(figsize=(9, 3))
labels = [EKMAN_ID2LABEL[i] for i in counts.index]
ax.bar(labels, counts.values, color=[EMOTION_COLORS[l] for l in labels])
ax.set_title('До аугментации')
plt.tight_layout(); plt.show()

## 2. Конфигурация аугментации

In [ ]:
# ── Настройки ────────────────────────────────────────────────────

# Метод: 'paraphrase' | 'backtranslation' | 'both'
# 'both' даёт наибольшее разнообразие, но требует ~2× больше времени
METHOD = 'both'

# Целевое число примеров на класс в train
# Только классы НИЖЕ этого значения будут аугментированы
TARGET_PER_CLASS = 3_000

# Сколько парафразов генерировать на каждое исходное предложение
N_VARIANTS = 3

# Batch size для инференса (уменьши если OOM)
BATCH_SIZE = 8

print(f'Метод: {METHOD}')
print(f'Цель:  {TARGET_PER_CLASS} примеров на класс')
print('\nКлассы которые будут аугментированы:')
for lid, cnt in counts.items():
    if cnt < TARGET_PER_CLASS:
        needed = TARGET_PER_CLASS - cnt
        print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>5} → +{needed}')

## 3. Запуск аугментации

Ориентировочное время на T4:
- `paraphrase` ~500 текстов × 3 варианта → **~5-10 мин**
- `backtranslation` ~500 текстов → **~3-5 мин**
- `both` → **~10-15 мин** суммарно

In [ ]:
from src.augmentation import augment_rare_classes

print('Запуск аугментации...')
ds_aug = augment_rare_classes(
    dataset=ds,
    label_names=EKMAN_LABEL_NAMES,
    target_per_class=TARGET_PER_CLASS,
    method=METHOD,
    n_variants=N_VARIANTS,
    batch_size=BATCH_SIZE,
    seed=42,
)
print('\nАугментация завершена!')

## 4. Визуализация результата

In [ ]:
train_aug_df = ds_aug['train'].to_pandas()
counts_aug = train_aug_df['label'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (cnt, title) in zip(axes, [(counts, 'До аугментации'), (counts_aug, 'После аугментации')]):
    labels = [EKMAN_ID2LABEL[i] for i in cnt.index]
    ax.bar(labels, cnt.values, color=[EMOTION_COLORS[l] for l in labels])
    ax.set_title(f'{title}\n(всего: {cnt.sum():,})')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/augmentation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nСравнение:')
print(f'{"Класс":<12}  {"До":>8}  {"После":>8}  {"Δ":>8}')
print('-' * 45)
for lid in range(len(EKMAN_LABEL_NAMES)):
    before = counts.get(lid, 0)
    after  = counts_aug.get(lid, 0)
    print(f'{EKMAN_LABEL_NAMES[lid]:<12}  {before:>8,}  {after:>8,}  +{after-before:>6,}')

## 5. Проверка качества аугментации

In [ ]:
# Посмотрим на примеры аугментированных текстов
orig_texts = set(ds['train'].to_pandas()['text'].tolist())
new_df = train_aug_df[~train_aug_df['text'].isin(orig_texts)]

print(f'Новых примеров добавлено: {len(new_df):,}\n')

for lid in range(len(EKMAN_LABEL_NAMES)):
    class_new = new_df[new_df['label'] == lid]
    if len(class_new) == 0:
        continue
    print(f'── {EKMAN_LABEL_NAMES[lid].upper()} (новых: {len(class_new)}) ──')
    for _, row in class_new.head(3).iterrows():
        print(f'  {row["text"][:120]}')
    print()

## 6. Сохранение

In [ ]:
ds_aug.save_to_disk(AUG_DATA_PATH)
print(f'Аугментированный датасет сохранён: {AUG_DATA_PATH}')
print('\nИспользование в ноутбуке 07:')
print(f'  Замени S1_DATA_PATH = \'{AUG_DATA_PATH}\'')
print('  (или просто убедись что файл называется stage1_data_augmented)')

# Статистика по сплитам
for split in ds_aug:
    print(f'  {split:12s}: {len(ds_aug[split]):,}')

## 7. Аугментация Stage-2 (чистый корпус)

Для Stage-2 нужно особенно аккуратно — датасет маленький, а disgust/surprise очень мало.

In [ ]:
S2_DATA_PATH = f'{WORKING_DIR}/stage2_data'
S2_AUG_PATH  = f'{WORKING_DIR}/stage2_data_augmented'

if os.path.exists(S2_DATA_PATH):
    ds2 = load_from_disk(S2_DATA_PATH)
    print('Stage-2 загружен:')
    for split in ds2:
        print(f'  {split:12s}: {len(ds2[split]):,}')

    print('\nРаспределение в Stage-2 train:')
    df2 = ds2['train'].to_pandas()
    for lid, cnt in df2['label'].value_counts().sort_index().items():
        print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>5,}')

    print('\nАугментация Stage-2...')
    ds2_aug = augment_rare_classes(
        dataset=ds2,
        label_names=EKMAN_LABEL_NAMES,
        target_per_class=400,   # небольшая цель — датасет намеренно маленький
        method='both',
        n_variants=5,           # больше вариантов, т.к. исходников очень мало
        batch_size=BATCH_SIZE,
        seed=42,
    )
    ds2_aug.save_to_disk(S2_AUG_PATH)
    print(f'Stage-2 augmented сохранён: {S2_AUG_PATH}')
else:
    print(f'Stage-2 не найден по пути {S2_DATA_PATH}')
    print('Запусти сначала ячейки загрузки данных в ноутбуке 07.')